In [18]:
import torch
import numpy as np
import torch.nn as nn

# TODO: Maybe use nn.Sequential for each decoder/decoder Block?

class UNetEncoderBlock(nn.Module):

    def __init__(self, input_channels, output_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(input_channels, output_channels, 3)
        self.conv2 = nn.Conv2d(output_channels, output_channels, 3)
        self.activation = nn.ReLU(inplace=True)
        self.downsample = nn.MaxPool2d(2, 2)

    def forward(self, x):
        x = self.conv1(x)
        x = self.activation(x)
        x = self.conv2(x)
        x = self.activation(x)
        x = self.downsample(x)

class UNetDecoderBlock(nn.Module):

    def __init__(self, input_channels, output_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(input_channels, output_channels, 3)
        self.conv2 = nn.Conv2d(output_channels, output_channels, 3)
        self.activation = nn.ReLU(inplace=True)
        self.upsample = nn.ConvTranspose2d(output_channels, output_channels // 2, 2, 2)

    def forward(self, x):
        x = self.conv1(x)
        x = self.activation(x)
        x = self.conv2(x)
        x = self.activation(x)
        x = self.upsample(x)

class UNet(nn.Module):

    def __init__(self):
        super().__init__()

        # Encoder layers
        self.encoder1 = UNetEncoderBlock(1, 64)
        self.encoder2 = UNetEncoderBlock(64, 128)
        self.encoder3 = UNetEncoderBlock(128, 256)
        self.encoder4 = UNetEncoderBlock(256, 512)

        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(512, 1024, 3),
            nn.ReLU(inplace=True),
            nn.Conv2d(1024, 1024, 3),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(1024, 512, 2, 2)
        )

        # Decoder layers
        self.decoder1 = UNetDecoderBlock(1024, 512)
        self.decoder2 = UNetDecoderBlock(512, 256)
        self.decoder3 = UNetDecoderBlock(256, 128)
        self.decoder4 = UNetDecoderBlock(128, 64)

        # Final Convolutional Layer
        self.output = nn.Conv2d(64, 4, 1)
        

    def forward(self, x):

        # Run the encoder
        o1 = self.encoder1(x)
        o2 = self.encoder2(o1)
        o3 = self.encoder3(o2)
        o4 = self.encoder4(o4)

        # Run the bottleneck
        x = self.bottleneck(o4)

        torch.cat()

        # Run the decoder
        x = self.decoder1(torch.cat((o4, x)))
        x = self.decoder2(torch.cat((o3, x)))
        x = self.decoder3(torch.cat((o2, x)))
        x = self.decoder4(torch.cat((o1, x)))

        # The final output segmentation with 4 classes
        return self.output(x)


In [20]:
unet = UNet()

grayscale_array = np.random.randint(0, 256, size=(572, 572), dtype=np.uint8)

unet(grayscale_array)

TypeError: conv2d() received an invalid combination of arguments - got (numpy.ndarray, Parameter, Parameter, tuple, tuple, tuple, int), but expected one of:
 * (Tensor input, Tensor weight, Tensor bias = None, tuple of ints stride = 1, tuple of ints padding = 0, tuple of ints dilation = 1, int groups = 1)
      didn't match because some of the arguments have invalid types: (!numpy.ndarray!, !Parameter!, !Parameter!, !tuple of (int, int)!, !tuple of (int, int)!, !tuple of (int, int)!, !int!)
 * (Tensor input, Tensor weight, Tensor bias = None, tuple of ints stride = 1, str padding = "valid", tuple of ints dilation = 1, int groups = 1)
      didn't match because some of the arguments have invalid types: (!numpy.ndarray!, !Parameter!, !Parameter!, !tuple of (int, int)!, !tuple of (int, int)!, !tuple of (int, int)!, !int!)
